# aging_transformation

> 对应代码：`EconometricsCode/code_in_notes/Chap.8/aging_transformation.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.8`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.8")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


我们读入跨国宏观数据，并生成人均 GDP 的对数及其平方项。把水平项和平方项同时放入回归，是为了刻画制造业就业份额随经济发展水平变化的非线性（倒 U 形或 U 形）关系。

In [2]:
%%stata
use ../datasets/aging_transformation_macro.dta, clear
// 不中心化
gen log_gdp_pc  = log(gdp_pc)
gen log_gdp_pc2 = log_gdp_pc^2


. use ../datasets/aging_transformation_macro.dta, clear

. // 不中心化
. gen log_gdp_pc  = log(gdp_pc)

. gen log_gdp_pc2 = log_gdp_pc^2

. 


我们首先估计一个含平方项的双向固定效应模型：被解释变量是制造业就业份额，解释变量包括对数人均 GDP 及其平方项，并控制 65 岁以上人口占比（老龄化程度）。reghdfe 的 absorb(code) 吸收国家固定效应，控制各国不随时间变化的异质性；vce(cl code) 在国家层面聚类标准误，允许同一国家内误差相关。

In [3]:
%%stata
reghdfe hw_share_man log_gdp_pc log_gdp_pc2 share_65plus, a(code) vce(cl code)

(MWFE estimator converged in 1 iterations)

HDFE Linear regression                            Number of obs   =        710
Absorbing 1 HDFE group                            F(   3,     18) =      15.39
Statistics robust to heteroskedasticity           Prob > F        =     0.0000
                                                  R-squared       =     0.6802
                                                  Adj R-squared   =     0.6704
                                                  Within R-sq.    =     0.5857
Number of clusters (code)    =         19         Root MSE        =     0.0270

                                  (Std. err. adjusted for 19 clusters in code)
------------------------------------------------------------------------------
             |               Robust
hw_share_man | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
  log_gdp_pc |   1.241136   .2313861     5.36   0.

接下来我们做一个去平均（demean）变换：先计算对数人均 GDP 的样本均值，再生成离差的平方。这样变换后，一次项系数的经济含义变为在样本均值处（而非原点处）的边际效应，解释更方便，且水平项与平方项之间的共线性也会下降。

In [4]:
%%stata
// 中心化
egen mean_log_gdp_pc = mean(log_gdp_pc)
gen dm_log_gdp_pc2 = (log_gdp_pc-mean_log_gdp_pc)^2


. // 中心化
. egen mean_log_gdp_pc = mean(log_gdp_pc)

. gen dm_log_gdp_pc2 = (log_gdp_pc-mean_log_gdp_pc)^2

. 


我们用去平均后的平方项重新回归。两个设定的拟合值完全等价，但此时一次项系数直接给出经济发展处于样本平均水平时制造业份额的边际效应，而平方项系数在两种设定下完全相同。

In [5]:
%%stata
reghdfe hw_share_man log_gdp_pc dm_log_gdp_pc2 share_65plus, a(code) vce(cl code)

(MWFE estimator converged in 1 iterations)

HDFE Linear regression                            Number of obs   =        710
Absorbing 1 HDFE group                            F(   3,     18) =      15.39
Statistics robust to heteroskedasticity           Prob > F        =     0.0000
                                                  R-squared       =     0.6802
                                                  Adj R-squared   =     0.6704
                                                  Within R-sq.    =     0.5857
Number of clusters (code)    =         19         Root MSE        =     0.0270

                                  (Std. err. adjusted for 19 clusters in code)
------------------------------------------------------------------------------
             |               Robust
hw_share_man | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
  log_gdp_pc |  -.0659487   .0300247    -2.20   0.